In [ ]:
!pip install ultralytics pycocotools opencv-python matplotlib wandb

In [ ]:
import os
import json
import shutil
import math
import numpy as np
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt
import torch
import wandb
from ultralytics import YOLO
# Updated import for callbacks
from ultralytics.utils import callbacks

In [ ]:
import wandb
from google.colab import userdata

try:
    # Retrieve the API key from Colab secrets
    wandb_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_key)
except Exception as e:
    print("Could not find WANDB_API_KEY in secrets. Falling back to interactive login.")
    wandb.login()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: michaelo-ponteski (ocr-pl-med) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
# ─── W&B initialisation ──────────────────────────────────────────────────
# Set WANDB_API_KEY env var or run `wandb login` once before training.
WANDB_PROJECT = "yolo-line-segmentation"
WANDB_RUN_NAME = "yolov8n-0.1"

run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "model": "yolov8n",
        "epochs": 100,
        "imgsz": 512,
        "batch": 64,
        "warmup_epochs": 5,
        "base_lr": 0.01,
        "momentum": 0.937,
        "weight_decay": 0.0005,
        "cosine_lr_decay": True,
        "grad_cosine_lr_reduction": 0.5,   # factor applied when grad cosine < 0
    },
)
cfg = run.config
print("W&B run initialised:", run.url)


W&B run initialised: https://wandb.ai/ocr-pl-med/yolo-line-segmentation/runs/dp5to83l


In [ ]:
# ─── Custom training callbacks ──────────────────────────────────────────────
import math
import torch
import wandb
from ultralytics.utils import callbacks

_prev_grads: dict = {}
_step_counter: int = 0
_current_epoch: int = 0
_train_callbacks_registered: bool = False
_smoothed_loss = None

def _on_train_batch_end(trainer):
    global _step_counter, _current_epoch, _smoothed_loss

    _step_counter += 1

    optimizer = trainer.optimizer
    epoch     = trainer.epoch
    _current_epoch = epoch

    # ── 1. Per-step LR warmup ─────────────────────────────────────────────────
    warmup_epochs = getattr(trainer.args, "warmup_epochs", 5)
    base_lr       = getattr(trainer.args, "lr0", 0.01)

    if epoch < warmup_epochs:
        epoch_len    = getattr(trainer, "epoch_len", None) or getattr(trainer, "nb", 1)
        steps_done   = epoch * epoch_len + getattr(trainer, "batch_i", 0) + 1
        warmup_factor = min(steps_done / (warmup_epochs * epoch_len), 1.0)
        for pg in optimizer.param_groups:
            pg["lr"] = base_lr * warmup_factor

    # ── 2. Training losses (Fixed Scale & Spikes) ─────────────────────────────
    log_dict = {
        "train/lr": optimizer.param_groups[0]["lr"],
        "step":     _step_counter,
    }

    if hasattr(trainer, "loss_items") and trainer.loss_items is not None:
        try:
            items = trainer.loss_items.detach().cpu().tolist()
            current_loss = sum(items)

            if current_loss > 0.0:
                log_dict["train/loss"] = current_loss

                if _smoothed_loss is None:
                    _smoothed_loss = current_loss
                else:
                    _smoothed_loss = 0.9 * _smoothed_loss + 0.1 * current_loss

                log_dict["train/loss_smoothed"] = _smoothed_loss

            names = ["box", "cls", "dfl"]
            for name, val in zip(names, items):
                if val > 0.0:
                    log_dict[f"train/loss_{name}"] = val
        except Exception:
            pass

    # ── 3. Omit logging for the first 10 batches ─────────────────────────────
    if _step_counter > 10:
        wandb.log(log_dict)

def _on_fit_epoch_end(trainer):
    metrics = {f"epoch/{k}": v for k, v in trainer.metrics.items()}
    metrics["epoch"] = trainer.epoch
    wandb.log(metrics)

def _on_train_end(trainer):
    # ── 5. Save Model Artifacts ───────────────────────────────────────────────
    try:
        from pathlib import Path
        artifact = wandb.Artifact(name=f"run_{wandb.run.id}_models", type="model")
        weights_dir = Path(trainer.save_dir) / "weights"
        if (weights_dir / "last.pt").exists():
            artifact.add_file(str(weights_dir / "last.pt"))
        if (weights_dir / "best.pt").exists():
            artifact.add_file(str(weights_dir / "best.pt"))
        wandb.log_artifact(artifact)
        print("Model artifacts (best.pt, last.pt) logged to W&B.")
    except Exception as e:
        print(f"Failed to log artifacts: {e}")

    wandb.finish()
    print("W&B run finished.")

def _register_train_callbacks():
    global _train_callbacks_registered
    if _train_callbacks_registered:
        return
    if hasattr(callbacks, 'default_callbacks'):
        callbacks.default_callbacks['on_train_batch_end'].append(_on_train_batch_end)
        callbacks.default_callbacks['on_fit_epoch_end'].append(_on_fit_epoch_end)
        callbacks.default_callbacks['on_train_end'].append(_on_train_end)
        _train_callbacks_registered = True
        print("Training callbacks registered.")

_register_train_callbacks()

Training callbacks registered.


In [ ]:
# ─── Task-specific validation metrics for handwritten line segmentation ────
import numpy as np
import cv2
import shutil
from pathlib import Path
import wandb
from ultralytics import YOLO
from ultralytics.utils import callbacks

_VAL_LABEL_DIR: str = None
_VAL_IMAGE_DIR: str = None
_val_callbacks_registered: bool = False

# ── Trackers for our custom best models ──
_best_missed_rate = float('inf')
_best_iou_median = -float('inf')

def _compute_iou_matrix(boxes_a, boxes_b):
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)))
    x1 = np.maximum(boxes_a[:, 0:1], boxes_b[:, 0])
    y1 = np.maximum(boxes_a[:, 1:2], boxes_b[:, 1])
    x2 = np.minimum(boxes_a[:, 2:3], boxes_b[:, 2])
    y2 = np.minimum(boxes_a[:, 3:4], boxes_b[:, 3])
    inter  = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)
    area_a = (boxes_a[:, 2] - boxes_a[:, 0]) * (boxes_a[:, 3] - boxes_a[:, 1])
    area_b = (boxes_b[:, 2] - boxes_b[:, 0]) * (boxes_b[:, 3] - boxes_b[:, 1])
    union  = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / union, 0.0)

def _xywhn_to_xyxy(boxes, w, h):
    cx, cy, bw, bh = boxes[:,0], boxes[:,1], boxes[:,2], boxes[:,3]
    return np.stack([(cx-bw/2)*w, (cy-bh/2)*h, (cx+bw/2)*w, (cy+bh/2)*h], axis=1)

def _ece(confs, corrects, n_bins=10):
    if len(confs) == 0: return 0.0
    ece = 0.0
    for lo, hi in zip(np.linspace(0,1,n_bins+1)[:-1], np.linspace(0,1,n_bins+1)[1:]):
        mask = (confs >= lo) & (confs < hi)
        if mask.sum() == 0: continue
        ece += mask.mean() * abs(corrects[mask].mean() - confs[mask].mean())
    return float(ece)

def compute_line_segmentation_metrics(
    val_label_dir, val_image_dir, yolo_model,
    iou_thresh=0.5, conf_thresh=0.25, max_images=200,
):
    label_files = sorted(Path(val_label_dir).glob("*.txt"))
    if max_images:
        label_files = label_files[:max_images]

    all_ious, all_confs, all_correct = [], [], []
    total_gt = total_pred = total_missed = total_splits = total_merges = 0
    cy_errors, ar_errors, pred_counts, gt_counts = [], [], [], []

    wandb_images = []

    for lf in label_files:
        img_path = None
        for ext in [".jpg", ".jpeg", ".png"]:
            p = Path(val_image_dir) / (lf.stem + ext)
            if p.exists(): img_path = p; break
        if img_path is None: continue

        img = cv2.imread(str(img_path))
        if img is None: continue
        H, W = img.shape[:2]

        rows = [l.split() for l in lf.read_text().strip().splitlines() if len(l.split()) >= 5]
        if not rows: continue
        gt_n  = np.array([[float(x) for x in r[1:5]] for r in rows])
        gt_xy = _xywhn_to_xyxy(gt_n, W, H)

        result = yolo_model(str(img_path), verbose=False, conf=conf_thresh)[0]
        preds  = result.boxes

        pred_xy = preds.xyxy.cpu().numpy() if preds is not None else []

        if len(wandb_images) < 10:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            pred_xyxyn = preds.xyxyn.cpu().numpy() if preds is not None and len(preds) > 0 else []
            pred_box_data = [
                {"position": {"minX": float(b[0]), "minY": float(b[1]), "maxX": float(b[2]), "maxY": float(b[3])}, "class_id": 0}
                for b in pred_xyxyn
            ]
            gt_box_data = []
            for b in gt_n:
                cx, cy, w, h = b[0], b[1], b[2], b[3]
                gt_box_data.append({
                    "position": {
                        "minX": float(max(0.0, cx - w/2)),
                        "minY": float(max(0.0, cy - h/2)),
                        "maxX": float(min(1.0, cx + w/2)),
                        "maxY": float(min(1.0, cy + h/2))
                    }, "class_id": 0
                })

            wandb_images.append(wandb.Image(img_rgb, boxes={
                "predictions": {"box_data": pred_box_data},
                "ground_truth": {"box_data": gt_box_data}
            }))

        if preds is None or len(preds) == 0:
            total_gt    += len(gt_xy); total_missed += len(gt_xy)
            gt_counts.append(len(gt_xy)); pred_counts.append(0)
            continue

        pred_conf = preds.conf.cpu().numpy()
        pred_n    = preds.xywhn.cpu().numpy()
        iou_mat   = _compute_iou_matrix(gt_xy, pred_xy)

        for i in range(len(gt_xy)):
            hits = np.where(iou_mat[i] >= iou_thresh)[0]
            if len(hits) == 0:
                total_missed += 1
            else:
                if len(hits) > 1: total_splits += 1
                all_ious.append(float(iou_mat[i, hits[np.argmax(iou_mat[i, hits])]]))

        for j in range(len(pred_xy)):
            hits  = np.where(iou_mat[:, j] >= iou_thresh)[0]
            is_tp = len(hits) >= 1
            all_confs.append(float(pred_conf[j]))
            all_correct.append(float(is_tp))
            if len(hits) > 1: total_merges += 1
            if is_tp:
                gi = hits[np.argmax(iou_mat[hits, j])]
                cy_errors.append(abs(float(pred_n[j,1]) - float(gt_n[gi,1])))
                gt_ar = gt_n[gi,2] / (gt_n[gi,3] + 1e-6)
                pr_ar = pred_n[j,2] / (pred_n[j,3] + 1e-6)
                ar_errors.append(abs(float(pr_ar) - float(gt_ar)))

        total_gt   += len(gt_xy);     total_pred += len(pred_xy)
        gt_counts.append(len(gt_xy)); pred_counts.append(len(pred_xy))

    ious    = np.array(all_ious)
    confs   = np.array(all_confs)
    correct = np.array(all_correct)
    counts  = np.abs(np.array(pred_counts) - np.array(gt_counts))

    m = {
        "val/iou_mean":             float(ious.mean())             if len(ious) else 0.0,
        "val/iou_median":           float(np.median(ious))         if len(ious) else 0.0,
        "val/iou_p10":              float(np.percentile(ious, 10)) if len(ious) else 0.0,
        "val/iou_p90":              float(np.percentile(ious, 90)) if len(ious) else 0.0,
        "val/missed_line_rate":     total_missed / max(total_gt,   1),
        "val/split_rate":           total_splits / max(total_gt,   1),
        "val/merge_rate":           total_merges / max(total_pred, 1),
        "val/median_cy_error_norm": float(np.median(cy_errors))    if cy_errors else 0.0,
        "val/median_ar_error":      float(np.median(ar_errors))    if ar_errors else 0.0,
        "val/line_count_acc_exact": float((counts == 0).mean())    if len(counts) else 0.0,
        "val/line_count_acc_pm1":   float((counts <= 1).mean())    if len(counts) else 0.0,
        "val/confidence_ece":       _ece(confs, correct),
    }

    if len(ious) > 0:
        m["val/iou_histogram"] = wandb.Histogram(ious.tolist())
    if wandb_images:
        m["val/predictions"] = wandb_images

    return m

def _on_fit_epoch_end_custom(trainer):
    global _best_missed_rate, _best_iou_median
    if _VAL_LABEL_DIR is None:
        return

    try:
        weights_path = getattr(trainer, 'last', None)
        if not weights_path or not Path(weights_path).exists():
            weights_path = Path(trainer.save_dir) / 'weights' / 'last.pt'

        if not Path(weights_path).exists():
            print("  [Val] Skipping validation: last.pt not found yet.")
            return

        yolo_val = YOLO(str(weights_path))
        extra = compute_line_segmentation_metrics(
            val_label_dir=_VAL_LABEL_DIR,
            val_image_dir=_VAL_IMAGE_DIR,
            yolo_model=yolo_val,
        )
        extra["epoch"] = trainer.epoch
        wandb.log(extra)

        # ─── Artifact Saving & Custom "Best" Logic ─────────────────────────────
        weights_dir = Path(trainer.save_dir) / 'weights'
        last_pt = weights_dir / 'last.pt'
        best_missed_pt = weights_dir / 'best_missed_line_rate.pt'
        best_iou_pt = weights_dir / 'best_iou_median.pt'

        current_missed = extra.get("val/missed_line_rate", float('inf'))
        current_iou = extra.get("val/iou_median", -float('inf'))

        improved_missed = False
        improved_iou = False

        # Check if we beat the missed line rate (lower is better)
        if current_missed < _best_missed_rate:
            _best_missed_rate = current_missed
            shutil.copy(str(last_pt), str(best_missed_pt))
            improved_missed = True

        # Check if we beat the IoU median (higher is better)
        if current_iou > _best_iou_median:
            _best_iou_median = current_iou
            shutil.copy(str(last_pt), str(best_iou_pt))
            improved_iou = True

        # Log models to W&B
        artifact = wandb.Artifact(name=f"run_{wandb.run.id}_models", type="model")
        artifact.add_file(str(last_pt), name="last.pt")

        if best_missed_pt.exists():
            artifact.add_file(str(best_missed_pt), name="best_missed_line_rate.pt")
        if best_iou_pt.exists():
            artifact.add_file(str(best_iou_pt), name="best_iou_median.pt")

        # Add dynamic W&B aliases so you can easily pull the best ones later
        aliases = ["latest", f"epoch_{trainer.epoch}"]
        if improved_missed: aliases.append("best_missed_line_rate")
        if improved_iou: aliases.append("best_iou_median")

        wandb.log_artifact(artifact, aliases=aliases)

        # Print summary
        for k in ["val/missed_line_rate", "val/split_rate", "val/merge_rate",
                  "val/iou_median", "val/line_count_acc_exact"]:
            if k in extra:
                print(f"  {k}: {extra[k]:.4f}")

        print(f"  [Artifacts] Saved to W&B. Best Missed: {_best_missed_rate:.4f} | Best IoU: {_best_iou_median:.4f}")

    except Exception:
        import traceback; traceback.print_exc()

def _register_val_callbacks():
    global _val_callbacks_registered
    if _val_callbacks_registered:
        return
    if hasattr(callbacks, 'default_callbacks'):
        callbacks.default_callbacks['on_fit_epoch_end'].append(_on_fit_epoch_end_custom)
        _val_callbacks_registered = True
        print("Task-specific val callbacks registered.")

_register_val_callbacks()

Task-specific val callbacks registered.


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("michaeloponteski/handwritten-lines-coco")

print("Path to dataset files:", path)
path = Path(path) / 'dataset'

Using Colab cache for faster access to the 'handwritten-lines-coco' dataset.
Path to dataset files: /kaggle/input/handwritten-lines-coco


In [ ]:
IMAGES_DIR = path / 'images'
COCO_JSON = path / "lines_coco.json"

YOLO_DIR = "yolo_dataset"

In [ ]:
def convert_coco_to_yolo(images_dir, coco_json, output_dir):
    with open(coco_json, "r", encoding="utf-8") as f:
        coco = json.load(f)

    os.makedirs(output_dir, exist_ok=True)
    images_out = os.path.join(output_dir, "images")
    labels_out = os.path.join(output_dir, "labels")
    os.makedirs(images_out, exist_ok=True)
    os.makedirs(labels_out, exist_ok=True)

    coco_root = os.path.dirname(os.path.abspath(coco_json))
    images_info = {img["id"]: img for img in coco.get("images", [])}

    categories = sorted(coco.get("categories", []), key=lambda c: c["id"])
    cat_id_to_idx = {cat["id"]: idx for idx, cat in enumerate(categories)}
    if not cat_id_to_idx:
        cat_id_to_idx = {1: 0}

    annotations = {}
    for ann in coco.get("annotations", []):
        img_id = ann.get("image_id")
        annotations.setdefault(img_id, []).append(ann)

    def resolve_image_path(file_name):
        base_name = os.path.basename(file_name)
        candidates = [
            file_name,
            os.path.join(coco_root, file_name),
            os.path.join(images_dir, file_name),
            os.path.join(images_dir, base_name),
            os.path.join(coco_root, images_dir, base_name),
        ]

        seen = set()
        for candidate in candidates:
            normalized = os.path.normpath(candidate)
            if normalized in seen:
                continue
            seen.add(normalized)
            if os.path.exists(normalized):
                return normalized
        return None

    copied_images = 0
    missing_images = 0
    written_labels = 0
    skipped_boxes = 0

    for img_id, img_info in tqdm(images_info.items(), desc="COCO -> YOLO"):
        file_name = img_info["file_name"]
        base_name = os.path.basename(file_name)
        width = float(img_info["width"])
        height = float(img_info["height"])

        if width <= 0 or height <= 0:
            missing_images += 1
            continue

        src_img_path = resolve_image_path(file_name)
        if src_img_path is None:
            missing_images += 1
            continue

        dst_img_path = os.path.join(images_out, base_name)
        shutil.copy(src_img_path, dst_img_path)
        copied_images += 1

        label_path = os.path.join(labels_out, os.path.splitext(base_name)[0] + ".txt")
        yolo_lines = []

        for ann in annotations.get(img_id, []):
            bbox = ann.get("bbox")
            if not bbox or len(bbox) != 4:
                skipped_boxes += 1
                continue

            x, y, w, h = map(float, bbox)
            if w <= 0 or h <= 0:
                skipped_boxes += 1
                continue

            x_center = (x + w / 2.0) / width
            y_center = (y + h / 2.0) / height
            w_norm = w / width
            h_norm = h / height

            if w_norm <= 0 or h_norm <= 0:
                skipped_boxes += 1
                continue

            x_center = min(max(x_center, 0.0), 1.0)
            y_center = min(max(y_center, 0.0), 1.0)
            w_norm = min(max(w_norm, 0.0), 1.0)
            h_norm = min(max(h_norm, 0.0), 1.0)

            cls = cat_id_to_idx.get(ann.get("category_id"), 0)
            yolo_lines.append(
                f"{cls} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
            )

        with open(label_path, "w", encoding="utf-8") as f:
            f.write("\n".join(yolo_lines))
        written_labels += 1

    print(f"Images copied: {copied_images}")
    print(f"Label files written: {written_labels}")
    print(f"Missing images in COCO paths: {missing_images}")
    print(f"Skipped invalid boxes: {skipped_boxes}")

In [ ]:
convert_coco_to_yolo(IMAGES_DIR, COCO_JSON, YOLO_DIR)

COCO -> YOLO: 100%|██████████| 3086/3086 [00:53<00:00, 57.34it/s]

Images copied: 3086
Label files written: 3086
Missing images in COCO paths: 0
Skipped invalid boxes: 0


In [ ]:
import random


def split_dataset(yolo_dir, val_ratio=0.2):
    images_path = os.path.join(yolo_dir, "images")
    labels_path = os.path.join(yolo_dir, "labels")

    train_img = os.path.join(yolo_dir, "train/images")
    train_lbl = os.path.join(yolo_dir, "train/labels")
    val_img = os.path.join(yolo_dir, "val/images")
    val_lbl = os.path.join(yolo_dir, "val/labels")

    for p in [train_img, train_lbl, val_img, val_lbl]:
        os.makedirs(p, exist_ok=True)

    files = os.listdir(images_path)
    random.shuffle(files)

    split = int(len(files) * (1 - val_ratio))

    train_files = files[:split]
    val_files = files[split:]

    def move(files, img_dst, lbl_dst):
        for f in files:
            shutil.move(os.path.join(images_path, f), os.path.join(img_dst, f))
            lbl = f.replace(".jpg", ".txt").replace(".png", ".txt")
            shutil.move(os.path.join(labels_path, lbl), os.path.join(lbl_dst, lbl))

    move(train_files, train_img, train_lbl)
    move(val_files, val_img, val_lbl)

In [ ]:
split_dataset(YOLO_DIR)

In [ ]:
# ─── Set val dirs for task-specific metrics ────────────────────────────────
# FIX: plain global assignment — all notebook cells share one namespace,
# so _VAL_LABEL_DIR written here is visible inside the metrics cell above.
# (The old __main__._VAL_LABEL_DIR = ... trick did NOT reach those globals.)
_VAL_LABEL_DIR = os.path.join(YOLO_DIR, "val", "labels")
_VAL_IMAGE_DIR = os.path.join(YOLO_DIR, "val", "images")
print("Val dirs set:")
print("  labels:", _VAL_LABEL_DIR)
print("  images:", _VAL_IMAGE_DIR)


Val dirs set:
  labels: yolo_dataset/val/labels
  images: yolo_dataset/val/images


In [ ]:
data_yaml = f"""
path: {YOLO_DIR}
train: train/images
val: val/images

names:
  0: line
"""

with open("data.yaml", "w") as f:
    f.write(data_yaml)

In [ ]:
# ─── Training ────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else (
    "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Training on device: {device}")

model = YOLO("yolov8n.pt")

model.train(
    data="data.yaml",
    epochs=cfg.epochs,
    imgsz=cfg.imgsz,
    batch=cfg.batch,          # 64 — larger batch for more stable gradients
    device=device,
    # ── Learning rate & scheduler ──────────────────────────────────────────
    lr0=cfg.base_lr,          # initial LR (warmup ramps up to this)
    cos_lr=cfg.cosine_lr_decay,   # cosine LR decay after warmup
    warmup_epochs=cfg.warmup_epochs,  # linear warmup handled both by YOLO
                                       # and our custom callback above
    # ── Momentum & regularisation ──────────────────────────────────────────
    momentum=cfg.momentum,    # SGD momentum (0.937 is YOLOv8 default)
    weight_decay=cfg.weight_decay,
    # ── Logging ───────────────────────────────────────────────────────────
    project="yolo-line-segmentation",
    name=WANDB_RUN_NAME,
    exist_ok=True,
)


Training on device: cuda
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n-0.1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patien

  val/missed_line_rate: 0.5814
  val/split_rate: 0.0704
  val/merge_rate: 0.0006
  val/iou_median: 0.7498
  val/line_count_acc_exact: 0.0300
  [Artifacts] Saved to W&B. Best Missed: 0.5814 | Best IoU: 0.7498

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      6.12G      1.435     0.9394      1.233        866        512: 100% ━━━━━━━━━━━━ 39/39 2.6s/it 1:41
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.3s/it 6.6s
                   all        618       8265      0.834       0.89       0.91      0.471


  val/missed_line_rate: 0.0497
  val/split_rate: 0.3339
  val/merge_rate: 0.0064
  val/iou_median: 0.8003
  val/line_count_acc_exact: 0.0400
  [Artifacts] Saved to W&B. Best Missed: 0.0497 | Best IoU: 0.8003

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      6.13G      1.295     0.8512      1.127       1115        512: 100% ━━━━━━━━━━━━ 39/39 2.7s/it 1:45
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.5s/it 7.5s
                   all        618       8265      0.848      0.916      0.931       0.51


  val/missed_line_rate: 0.0141
  val/split_rate: 0.4290
  val/merge_rate: 0.0073
  val/iou_median: 0.8278
  val/line_count_acc_exact: 0.0050
  [Artifacts] Saved to W&B. Best Missed: 0.0141 | Best IoU: 0.8278

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      6.14G      1.235      0.802      1.083       1026        512: 100% ━━━━━━━━━━━━ 39/39 2.6s/it 1:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.7s/it 8.7s
                   all        618       8265      0.904       0.95      0.969       0.49


  val/missed_line_rate: 0.0091
  val/split_rate: 0.2015
  val/merge_rate: 0.0112
  val/iou_median: 0.7858
  val/line_count_acc_exact: 0.1250
  [Artifacts] Saved to W&B. Best Missed: 0.0091 | Best IoU: 0.8278

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      5/100      6.15G      1.229     0.7919      1.069        781        512: 100% ━━━━━━━━━━━━ 39/39 2.6s/it 1:42
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.7s/it 8.3s
                   all        618       8265      0.906      0.936      0.961      0.562


...

  val/missed_line_rate: 0.0016
  val/split_rate: 0.0038
  val/merge_rate: 0.0003
  val/iou_median: 0.9478
  val/line_count_acc_exact: 0.7950
  [Artifacts] Saved to W&B. Best Missed: 0.0016 | Best IoU: 0.9482

100 epochs completed in 4.782 hours.
Optimizer stripped from /content/runs/detect/yolo-line-segmentation/yolov8n-0.1/weights/last.pt, 6.2MB
Optimizer stripped from /content/runs/detect/yolo-line-segmentation/yolov8n-0.1/weights/best.pt, 6.2MB

Validating /content/runs/detect/yolo-line-segmentation/yolov8n-0.1/weights/best.pt...
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 2.4s/it 12.0s
                   all        618       8265      0.996      0.996      0.994      0.859
Speed: 0.1ms preprocess, 1.5ms inference, 0.0ms loss, 4.2ms postprocess per image
R

  val/missed_line_rate: 0.0016
  val/split_rate: 0.0038
  val/merge_rate: 0.0003
  val/iou_median: 0.9478
  val/line_count_acc_exact: 0.7950
  [Artifacts] Saved to W&B. Best Missed: 0.0016 | Best IoU: 0.9482
Model artifacts (best.pt, last.pt) logged to W&B.


epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇████
epoch/metrics/mAP50(B),▁▆▇█████████████████████████████████████
epoch/metrics/mAP50-95(B),▁▂▁▃▂▅▅▆▆▆▇▇▇▆▇▇▇▇▇▇▇███████████████████
epoch/metrics/precision(B),▁▂▄▆▅▇▇▇▇▇▇██▇██▇███████████████████████
epoch/metrics/recall(B),▁▄▆▅▇██▇█▇▇█████████████████████████████
epoch/val/box_loss,█▇▇▆▆▃▅▄▄▃▄▃▃▂▂▃▂▂▂▂▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val/cls_loss,█▅▅▄▄▄▄▃▃▂▂▂▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val/dfl_loss,█▇▆▅▅▃▄▂▃▂▂▂▂▁▂▂▁▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇█
train/loss,██▇▆▆▅▅▅▄▅▄▄▄▃▄▄▄▄▄▃▄▄▃▃▃▃▃▃▃▃▂▃▃▂▃▁▂▁▁▁
+17,...


W&B run finished.


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7bd0cef21a60>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 